<a href="https://colab.research.google.com/github/ancestor9/2026_Fall_Learning-Langchain-AI-Agent/blob/main/excercise/Real_Time_Websearch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ddgs langchain langchain-groq langchain-huggingface langchain-community langchain-text-splitters -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 972.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
"""
Realtime Web Search Agent (Groq LLM + HuggingFace Embeddings 버전)
-------------------------------------------------------------------
원본 노트북(Realtime_Web_Search.ipynb)을 아래 2가지 조건으로 리팩토링:
  1. LLM: Google Gemini -> Groq (langchain_groq.ChatGroq)
  2. Embedding: langchain_huggingface.HuggingFaceEmbeddings 추가
     -> 스크래핑한 페이지 본문 중 쿼리와 의미적으로 가장 관련 있는
        문단만 골라 LLM에 전달하는 데 사용 (앞부분 4000자 단순 자르기를 대체)

필요 패키지:
    pip install ddgs langchain langchain-groq langchain-huggingface \
                langchain-community langchain-text-splitters \
                sentence-transformers python-dotenv numpy -U
"""

import os

import numpy as np
from dotenv import load_dotenv

from ddgs import DDGS
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings  # Updated import

from langchain.agents import create_agent


# ---------------------------------------------------------------------------
# 0. 환경 설정 & API 키
# ---------------------------------------------------------------------------
load_dotenv()  # 로컬 실행 시 .env 파일에서 환경변수 로드

os.environ["USER_AGENT"] = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
)

# Google Colab 환경이면 userdata에서, 그 외(로컬/서버)에서는 .env 또는
# 이미 설정된 환경변수에서 GROQ_API_KEY를 읽어옵니다.
try:
    from google.colab import userdata  # type: ignore

    os.environ["GROQ_API_KEY"] = userdata.get("groq")
except ImportError:
    if not os.environ.get("GROQ_API_KEY"):
        raise EnvironmentError(
            "GROQ_API_KEY가 설정되어 있지 않습니다. "
            ".env 파일에 GROQ_API_KEY=your_key 형태로 추가하거나 "
            "환경변수로 미리 설정해주세요."
        )

/tmp/ipykernel_3141/839281243.py:22: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader


In [ ]:
import os
import numpy as np
from dotenv import load_dotenv
from ddgs import DDGS
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

# langchain 1.0+ 에서는 AgentExecutor / create_tool_calling_agent가 제거되었고,
# LangGraph 기반의 create_agent 하나로 통합되었습니다.
from langchain.agents import create_agent


# ---------------------------------------------------------------------------
# 1. 임베딩 & 유틸리티
# ---------------------------------------------------------------------------
os.environ["HF_HOME"] = "./hf_cache"
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
_text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

def _cosine_similarity(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10))

def _get_relevant_chunks(query, page_content, top_k=3):
    chunks = _text_splitter.split_text(page_content)
    if not chunks:
        return ""
    query_vector = embeddings.embed_query(query)
    chunk_vectors = embeddings.embed_documents(chunks)
    scored = [(_cosine_similarity(np.array(query_vector), np.array(vec)), chunk) for vec, chunk in zip(chunk_vectors, chunks)]
    scored.sort(key=lambda x: x[0], reverse=True)
    return "\n\n".join(chunk for _, chunk in scored[:top_k])

# ---------------------------------------------------------------------------
# 2. 웹 검색 도구
# ---------------------------------------------------------------------------
@tool
def web_search(query: str, num_results: int = 3) -> str:
    """실시간 웹 검색을 수행하고 콘텐츠를 가져옵니다."""
    print(f"🔎 검색 중: '{query}'...")
    try:
        results = DDGS().text(query=query, max_results=num_results, region="us-en")
        if not results:
            return f"'{query}'에 대한 결과가 없습니다."
        formatted = []
        for i, res in enumerate(results, 1):
            href = res.get("href", "")
            try:
                loader = WebBaseLoader(href)
                loader.requests_kwargs = {"timeout": 5}
                docs = loader.load()
                content = "\n".join(d.page_content for d in docs)
                relevant = _get_relevant_chunks(query, content)
                body = f"관련 내용:\n{relevant}" if relevant else f"요약:\n{res.get('body','')}"
            except Exception:
                body = f"요약:\n{res.get('body','')}"
            formatted.append(f"결과 {i}: {res.get('title')}\nURL: {href}\n{body}")
        return "\n---\n".join(formatted)
    except Exception as e:
        return f"검색 오류: {e}"

# ---------------------------------------------------------------------------
# 3. LLM & 에이전트 생성 (create_agent 사용)
# ---------------------------------------------------------------------------
# llama-3.3-70b-versatile은 최근 Groq에서 tool call을 "<function=...>" 같은
# 잘못된 포맷으로 생성해 tool_use_failed 오류를 자주 일으키고, Groq 측에서도
# 해당 모델을 단계적으로 지원 종료(deprecate)하는 중입니다.
# tool use에 최적화된 openai/gpt-oss-120b(또는 경량 openai/gpt-oss-20b)로 대체합니다.
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)
tools = [web_search]

system_prompt = "당신은 전문 리서치 에이전트입니다. 항상 web_search를 사용하여 사실을 확인하고 URL 출처를 밝히세요."

agent = create_agent(model=llm, tools=tools, system_prompt=system_prompt)

# ---------------------------------------------------------------------------
# 4. 실행
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    query = "최신 AI 기술의 주요 발전 사항은 무엇인가요?"
    try:
        result = agent.invoke({"messages": [HumanMessage(content=query)]})
        final_message = result["messages"][-1]
        content = final_message.content

        # Groq 응답이 문자열이거나 [{'type': 'text', 'text': ...}] 리스트 형태일 수 있음
        if isinstance(content, list):
            text_parts = [part.get("text", "") for part in content if isinstance(part, dict)]
            print("\n최종 결과:\n", "\n".join(text_parts) if text_parts else content)
        else:
            print("\n최종 결과:\n", content)
    except Exception as e:
        print(f"오류 발생: {e}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

🔎 검색 중: '2024 major AI technology advancements foundation models multimodal diffusion models reinforcement learning hardware AI chips 2024 news'...

최종 결과:
 ## 최신 AI 기술 ‑ 핵심 발전 사항 (2023 ~ 2024)

아래는 2023‑2024년에 학계·산업에서 눈에 띄게 진전된 주요 기술 트렌드와 그 의미를 정리한 내용입니다. 모든 항목은 **실시간 웹 검색 결과**를 근거로 하며, 출처 URL을 함께 제시합니다.

| # | 기술·트렌드 | 핵심 내용 | 주요 사례·논문·제품 | 출처 |
|---|-------------|-----------|-------------------|------|
| 1 | **대형 언어·멀티모달 파운데이션 모델** | • 1조‑10조 파라미터 규모의 LLM이 “범용 지식·추론” 능력을 확보하고, 텍스트·이미지·오디오·비디오를 동시에 처리하는 멀티모달 아키텍처가 급속히 확산.<br>• 모델‑중심 파이프라인 → “프롬프트 엔지니어링”, “인-컨텍스트 학습”, “리트리벌‑증강 생성(RAG)”이 표준화됨. | • **Google Gemini 2.0** (텍스트·이미지·비디오·에이전트 기능 통합) <br>• **Meta LLaMA‑3**, **OpenAI GPT‑4o**, **Anthropic Claude‑3** 등 최신 LLM <br>• **Nature** 리뷰: 멀티모달 파운데이션 모델이 의료·생명과학에 미치는 파급 효과 (2024) | 【3†URL】 |
| 2 | **확산 모델( Diffusion Models )** | • 텍스트‑투‑이미지·텍스트‑투‑비디오·오디오·3D 메쉬 생성 등 “고품질 생성 AI”의 핵심 엔진으로 자리 잡음.<br>• “조건부 확산”, “텍스트‑비전‑조건부 사이클 디퓨전”, “도메인‑불변 확산” 등 다양한 변형이 발표돼 특수 도메인(의료 영상, 과학 시뮬레이션)에도 적용 가능. 